In [ ]:
# ─── Standard Library ───────────────────────────────────────────────────────────
from pathlib import Path
import heapq

# ─── Scientific Computing ───────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ─── Geospatial Processing ──────────────────────────────────────────────────────
import geopandas as gpd
from shapely.geometry import box

# ─── Network Analysis ───────────────────────────────────────────────────────────
import pandana

# ─── Mapping and Visualization ──────────────────────────────────────────────────
import matplotlib.pyplot as plt

# ─── Progress Monitoring ────────────────────────────────────────────────────────
from tqdm.notebook import tqdm

# ─── Custom Modules ────────────────────────────────────────────────────────────
import geo_util as gu

In [ ]:
# global definitions
_data_path = Path( './data/' )

In [ ]:
max_dist_threshold = 20_000  # in meters

In [ ]:
dist = pd.read_parquet(_data_path.joinpath(f'access_matrix_{max_dist_threshold}.parquet'))

In [ ]:
use_pop = gpd.read_parquet(_data_path.joinpath('population_aggregated_per_node.parquet'))

In [ ]:
use_banks = gpd.read_parquet(_data_path.joinpath('banks_aggregated_per_node.parquet'))

In [ ]:
dist_matrix = gu.SparseDistanceMatrix(
    dist,
    pop_col='pop_idx',
    poi_col='poi_idx',
    dist_col='total_dist'
)

In [ ]:
import maxcovering as mc

In [ ]:
radius = 15_000.0  # in meters

In [ ]:
IJ = dist_matrix.popToPOIsWithin(radius)

In [ ]:
I = IJ.keys()

In [ ]:
J = sorted( set().union(*IJ.values()) )

In [ ]:
f'{len(I)=:,} {len(J)=:,}'

In [ ]:
results = mc.OptimizeWithGurobipy(
    w=use_pop.Population.values,
    I=I,
    J=J,
    IJ=IJ,
    budget_list=range(100,301,50),
    parsimonious=False,
    progress=tqdm, 
    maxTimeInSeconds=5*60, 
    mipGap=1e-8, 
    trace=False, 
    already_open=[]
)

In [ ]:
results_df = pd.DataFrame.from_dict(results,orient='index')

In [ ]:
results_df.solution.apply(len)

In [ ]:
( results_df.value / use_pop.Population.sum() ).plot()

In [ ]:
sol = results_df.solution.values[-1]

In [ ]:
[m for m in dir(dist_matrix) if not m.startswith('_')]

In [ ]:
JI = dist_matrix.poiToPopsWithin(radius)

In [ ]:
set( sol ) - set( JI.keys() )

In [ ]:
served = sorted( set().union(*(JI[j] for j in sol if j in JI)) )

In [ ]:
len(served),len(use_pop),len(use_pop.loc[~use_pop.index.isin(served)])

In [ ]:
use_pop.loc[~use_pop.index.isin(served)].Population.sum()/use_pop.Population.sum()

In [ ]:
from folium import Map
from folium.plugins import HeatMap
import folium

# Subset unserved and reproject to EPSG:4326
unserved_pop = use_pop.loc[~use_pop.index.isin(served)].copy()
unserved_pop = unserved_pop.to_crs(epsg=4326)

# Check which column holds the weights
weight_column = 'Population'  # adjust this if it's named differently

# Build weighted heatmap data: [lat, lon, weight]
heat_data = [
    [geom.y, geom.x, weight]
    for geom, weight in zip(unserved_pop.geometry, unserved_pop[weight_column])
    if weight > 0
]

# Add to existing map or create new one
m = use_banks.loc[sol].explore(color='green',style_kwds={'radius': 3, 'fillOpacity': 0.5})

HeatMap(
    heat_data,
    radius=12,
    blur=8,
    max_zoom=12,
    name='Unserved Pop (Weighted Heatmap)'
).add_to(m)

folium.LayerControl().add_to(m)
m

In [ ]:
m